In [1]:
# !pip install tabulate
import pandas as pd
import os

def count_value_in_column_where_target_is_zero(df, column_name, value_to_count=1, target_column='target'):
    """
    Counts the number of times a specific value appears in a given column
    for rows where the target column is 0.
    """
    target_zero_df = df[df[target_column] == 0]
    count = (target_zero_df[column_name] == value_to_count).sum()
    return count

def count_value_in_column_where_target_is_one(df, column_name, value_to_count=1, target_column='target'):
    """
    Counts the number of times a specific value appears in a given column
    for rows where the target column is 1.
    """
    target_one_df = df[df[target_column] == 1]
    count = (target_one_df[column_name] == value_to_count).sum()
    return count

if __name__ == '__main__':
    # Load the data
    df = pd.read_csv('data_IUPACs.csv', index_col=[0])

    # Use a list to store dictionaries for each row of our final table
    results_data = []

    # Iterate through each column in the DataFrame
    for col in df.columns:
        # Skip the target column itself
        if col == 'target':
            continue

        # Count occurrences of 1 for target 0 and target 1
        # Using df.copy() in function calls is generally good practice to prevent
        # unexpected modifications if the functions were to alter the DataFrame.
        count_target_zero = count_value_in_column_where_target_is_zero(df.copy(), col)
        count_target_one = count_value_in_column_where_target_is_one(df.copy(), col)

        # Calculate the proportion of 1s where target is 1
        total_count = count_target_one + count_target_zero
        if total_count > 0:
            proportion = round((count_target_one / total_count) * 100, 2)
        else:
            proportion = 0.0  # Avoid division by zero if no 1s found in either target group

        # Append the results for the current column to our list
        results_data.append({
            'column': col,
            'proportion': proportion,
            'count_target_one': count_target_one,
            'count_target_zero': count_target_zero
        })

    # Create a Pandas DataFrame from the collected results
    results_df = pd.DataFrame(results_data)

    results_df = results_df[results_df['proportion'] > 99.00]
    
    # Sort the DataFrame by 'proportion' in descending order
    results_df = results_df.sort_values(by='count_target_one', ascending=False)

    results_df.to_csv('functional_groups.csv')

    print("\n--- Summarized Column Analysis ---")
    # Print the DataFrame in a markdown table format for clear display
    print(results_df.to_markdown(index=False))


--- Summarized Column Analysis ---
| column                                   |   proportion |   count_target_one |   count_target_zero |
|:-----------------------------------------|-------------:|-------------------:|--------------------:|
| iodo                                     |          100 |                 60 |                   0 |
| sulfanylideneprop                        |          100 |                 44 |                   0 |
| dioxonaphthalen                          |          100 |                 42 |                   0 |
| oxonaphthalen                            |          100 |                 37 |                   0 |
| ynoxyethoxy                              |          100 |                 32 |                   0 |
| phenylhydrazinylidene                    |          100 |                 24 |                   0 |
| dihydroindeno                            |          100 |                 24 |                   0 |
| epoxyisoindol                      

In [2]:
# Find similar rows in 'column' 
group_by_column = 'column'

# Define the aggregations, specifying the aggregations
summarized_values = results_df.groupby(group_by_column).agg(
    Count_target_1=('count_target_one', 'sum'),           # Sum of target 1 for each Analyzed_Column
    Count_target_0=('count_target_zero', 'sum'),          # Sum of target 2 for each Analyzed_Column
)

# Organise a descending order 
df_sorted_desc = summarized_values.sort_values(by='Count_target_1', ascending=False)

# Display the result 
print(f"\nSummarized DataFrame by '{group_by_column}':")
print(df_sorted_desc.shape)
print(df_sorted_desc.head(30))


Summarized DataFrame by 'column':
(1296, 2)
                       Count_target_1  Count_target_0
column                                               
iodo                               60               0
sulfanylideneprop                  44               0
dioxonaphthalen                    42               0
oxonaphthalen                      37               0
ynoxyethoxy                        32               0
phenylhydrazinylidene              24               0
dihydroindeno                      24               0
epoxyisoindol                      24               0
pyrazolidine                       18               0
dibenzo                            18               0
trihydroxyphenyl                   18               0
icosa                              17               0
diazapentacyclo                    17               0
anthracene                         17               0
hydroiodide                        17               0
methylpyran                        15

In [3]:
df =  pd.read_csv('data_CID_SID_IUPACs_targets.csv')

# Define the columns and the condition
column_to_get_value = 'UPAC'              # The column from which the values will be retreaved 
column_with_condition = 'iodo'  # The column to check if its value is 1
condition_value = 1                       # The specific value (1) 

# Filter the DataFrame and select the desired column
# This operation first filters the rows where 'is_active' is 1,
# Then selects the 'item_name' from those filtered rows.
result_series = df[df[column_with_condition] == condition_value][column_to_get_value]

# Print the result(s)
# print(f"Values in '{column_to_get_value}' where '{column_with_condition}' is '{condition_value}':")

if not result_series.empty:
    for value in result_series:
        print(value)
    # Alternatively, print as a list:
        # print(result_series.tolist())
else:
    print("No rows found where the condition is met.")

print("-" * 40)

3-[(7S,10S,13S)-7-carbamoyl-10-(hydroxymethyl)-20-iodo-18-nitro-9,12,15-trioxo-2,8,11,14-tetrazabicyclo[14.4.0]icosa-1(16),17,19-trien-13-yl]propanoic acid
(5S,8S,11S)-11-(hydroxymethyl)-8-[(4-hydroxyphenyl)methyl]-18-iodo-16-nitro-7,10,13-trioxo-2,6,9,12-tetrazabicyclo[12.4.0]octadeca-1(14),15,17-triene-5-carboxamide
(5S,8S,11S)-18-iodo-8-(2-methylpropyl)-16-nitro-7,10,13-trioxo-11-propan-2-yl-2-oxa-6,9,12-triazabicyclo[12.4.0]octadeca-1(14),15,17-triene-5-carboxamide
(5S,8S,11S)-11-(3-amino-3-oxopropyl)-18-iodo-8-methyl-16-nitro-7,10,13-trioxo-2,6,9,12-tetrazabicyclo[12.4.0]octadeca-1(14),15,17-triene-5-carboxamide
(10S)-4-(2-amino-2-oxoethyl)-18-iodo-20-nitro-2,5,11-trioxo-16-oxa-3,6,12-triazatricyclo[15.4.0.06,10]henicosa-1(17),18,20-triene-13-carboxamide
4-amino-3-[[(7S,10S,13S)-10-(hydroxymethyl)-20-iodo-18-nitro-9,12,15-trioxo-13-propan-2-yl-2,8,11,14-tetrazabicyclo[14.4.0]icosa-1(16),17,19-triene-7-carbonyl]amino]-4-oxobutanoic acid
(5S,11S)-11-[(1R)-1-hydroxyethyl]-18-iodo-16-